# Reflex Colab smoke test (free tier, ~5 min)

Proves this repo's collector + interchange on real NVIDIA hardware with **zero pip installs**. Open this notebook in Colab (Runtime > Change runtime type > GPU), then Run all.

What it does: probes the GPU, clones `MugiZer/reflex`, runs the stdlib-only test subset, and proves real hardware identity capture (the field every collected run is keyed on). Collection itself is the follow-up notebook.

What it does NOT do: install nsys, profile anything, spend quota. That's the next notebook.

In [ ]:
# Cell 1 — environment probe. Fails fast on wrong/missing GPU.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import platform; print(platform.python_version())"
import json, pathlib
pathlib.Path('/content/reflex_runs').mkdir(exist_ok=True)
print('probe ok — continue only if a T4/P100 (or better) is listed above')

In [ ]:
# Cell 2 — clone (public repo, no auth needed).
!test -d reflex || git clone --depth 1 https://github.com/MugiZer/reflex.git
%cd reflex
!git log --oneline -1

In [ ]:
# Cell 3 — smoke tests. Stdlib only: no pip install, no GPU code paths.
# (Full suite needs sklearn etc. — install requirements only if you want it.)
!python -m pytest tests/test_reflex_ledger.py tests/test_fakegpu.py tests/test_collect.py -q

In [ ]:
# Cell 4 — REAL hardware identity. The money cell: proves the machine
# yields comparable identity before any collection is attempted.
from reflex.collect import nvidia_smi_identity
ident = nvidia_smi_identity()
print('identity:', ident)
assert ident['hardware'] != 'unknown', 'no GPU identity — aborting (see runbook)'
print('identity ok — this dict travels in every run manifest')

In [ ]:
# Cell 5 — device contract (what Colab must provide per run):
#   device(fault: str, seed: int) -> {artifact_name: bytes}
# e.g. {'trace.json': <kineto JSON bytes>} and/or {'subset.db': <nsys sqlite bytes>}.
# The collector handles manifest/DONE/checksums/resume around it.
# Paste failures + the manifest JSON back to the repo owner.

In [ ]:
# Cell 6 — Drive backup (optional). Stage locally first (done above),
# single copy out; never profile onto Drive directly.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil, datetime
    dst = '/content/drive/MyDrive/reflex-colab/%s' % datetime.date.today().isoformat()
    shutil.copytree('/content/reflex_runs', dst, dirs_exist_ok=True)
    print('backed up to', dst)
except Exception as exc:
    print('drive backup skipped:', type(exc).__name__, exc)

## Next steps

1. Paste back: the identity dict from Cell 4, the pytest tail from Cell 3, and any failure.
2. Full collection matrix (all faults × seeds, <2h shards, checkpoint every run) is the follow-up notebook — it reuses `collect()` + `scan_todo()` resume, so a killed session loses at most one run.
3. Open in Colab directly: `https://colab.research.google.com/github/MugiZer/reflex/blob/main/colab/Reflex_Colab_Smoke.ipynb`